# Notebook 12: Serving Metrics

**Series:** LLM Systems & Inference  
**Prerequisites:** Notebooks 05 (KV cache), 11 (Continuous Batching)

---

## Background: Why LLM Metrics Are Weird

Traditional web service metrics are simple: latency (how long did one request take?) and throughput (how many requests per second?). These map cleanly onto the work: one HTTP request → one response → one latency measurement.

LLM inference breaks this model in a fundamental way. The "response" is a *stream of tokens* that arrives over time, not a single value. A user asking a simple question gets their answer in 3 tokens; a user asking for a 2,000-word essay waits for hundreds of tokens to stream out. The same model, the same hardware, the same batch size — but wildly different user experiences.

This created a measurement crisis in 2022–2023. Early LLM API providers were reporting confusing numbers: "20 tokens/second" could mean a fast model that streams quickly, or a slow model that batches heavily for throughput. Users didn't know what they were getting.

### The Three Metrics That Matter

The community converged on three canonical metrics:

**1. TTFT — Time to First Token**  
How long from "submit request" to "see the first token appear"? This is the dominant user-experience metric for interactive applications. A chatbot with 2-second TTFT feels sluggish even if it streams quickly afterward. TTFT is dominated by prefill time (processing the input tokens) plus any queuing delay.

**2. TBT — Time Between Tokens (also called TPOT: Time Per Output Token)**  
Once streaming starts, how fast do tokens arrive? This is the "streaming speed" metric. At 30ms/token, output streams at roughly 33 tokens/second — fast enough to feel natural to humans (average reading speed is ~4-5 tokens/second, but people notice lag above ~10 tokens/second). TBT is dominated by the decode phase.

**3. Throughput**  
How many total tokens can the system generate per second across *all* concurrent requests? This is the operator's metric — it determines how many users you can serve with a given GPU budget. Throughput and TTFT are in direct tension: serving more users simultaneously increases TTFT (more queuing) but improves GPU utilization.

### The Latency-Throughput Tradeoff

Every production LLM serving system lives on a Pareto frontier: you can improve latency (serve fewer concurrent users) or throughput (accept worse latency to serve more users), but not both simultaneously. Understanding *where* you are on that frontier — and where you want to be — is the job of serving metrics.

This is exactly what `benchpress` (from the project backlog) measures: not just raw speed, but the full TTFT/TBT/throughput profile under realistic load, with statistical confidence intervals.

### How the Community Measures This

- **vLLM benchmarks**: The gold standard. `benchmark_serving.py` in the vLLM repo sends realistic request streams and reports P50/P99 TTFT, TBT, and throughput.
- **LMSys Chatbot Arena**: Measures user-facing latency at scale — real-world distribution of prompt/output lengths
- **MLPerf Inference**: Industry benchmark, but targets datacenter hardware at specific operating points, not the full frontier
- **llm-bench, llama-bench**: Simple throughput benchmarks for local inference; don't capture latency distribution

---

## What You'll Build

1. **Metric definitions and measurement harness** — TTFT, TBT, E2E latency
2. **Request stream simulator** — realistic load generation with Poisson arrivals
3. **Latency-throughput frontier** — sweep over batch sizes and arrival rates
4. **Percentile analysis** — P50, P90, P99 — why tail latency matters more than mean
5. **Statistical confidence intervals** — bootstrap CI for metric estimates
6. **`benchpress` prototype** — end-to-end measurement harness for local models

In [ ]:
!pip install -q numpy matplotlib scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from dataclasses import dataclass, field
from typing import Optional
from collections import deque
from scipy import stats as scipy_stats
import time

np.random.seed(42)
print("Setup complete.")

## Part 1: Metric Definitions

Let's formalize the three core metrics and their components.

In [ ]:
@dataclass
class TokenEvent:
    """A single token generation event — the atomic unit of measurement."""
    req_id: int
    token_index: int       # 0 = first generated token, 1 = second, etc.
    timestamp_ms: float    # wall clock time when this token was emitted
    is_first_token: bool   # True if this is the first output token


@dataclass
class RequestMetrics:
    """Complete timing metrics for a single request."""
    req_id: int
    prompt_len: int
    output_len: int
    submit_time_ms: float      # when user submitted the request
    first_token_time_ms: float # when first output token was generated
    last_token_time_ms: float  # when last output token was generated
    
    @property
    def ttft_ms(self) -> float:
        """Time to first token (includes queuing + prefill)."""
        return self.first_token_time_ms - self.submit_time_ms
    
    @property
    def tbt_ms(self) -> Optional[float]:
        """Average time between tokens (decode phase only)."""
        if self.output_len <= 1:
            return None
        decode_duration = self.last_token_time_ms - self.first_token_time_ms
        return decode_duration / max(1, self.output_len - 1)
    
    @property
    def e2e_latency_ms(self) -> float:
        """End-to-end latency: submit to last token."""
        return self.last_token_time_ms - self.submit_time_ms
    
    @property
    def normalized_latency(self) -> float:
        """E2E latency normalized by output length (ms per output token)."""
        return self.e2e_latency_ms / max(1, self.output_len)


@dataclass 
class ServingMetrics:
    """Aggregate metrics for a serving benchmark run."""
    requests: list[RequestMetrics] = field(default_factory=list)
    total_duration_ms: float = 0.0
    
    @property
    def throughput_tps(self) -> float:
        """Output tokens per second across all requests."""
        total_tokens = sum(r.output_len for r in self.requests)
        return total_tokens / (self.total_duration_ms / 1000.0)
    
    @property
    def request_rate(self) -> float:
        """Completed requests per second."""
        return len(self.requests) / (self.total_duration_ms / 1000.0)
    
    def ttft_percentile(self, p: float) -> float:
        return np.percentile([r.ttft_ms for r in self.requests], p)
    
    def tbt_percentile(self, p: float) -> float:
        tbts = [r.tbt_ms for r in self.requests if r.tbt_ms is not None]
        return np.percentile(tbts, p) if tbts else 0.0
    
    def e2e_percentile(self, p: float) -> float:
        return np.percentile([r.e2e_latency_ms for r in self.requests], p)
    
    def summary(self) -> str:
        lines = [
            f"Requests:           {len(self.requests)}",
            f"Duration:           {self.total_duration_ms/1000:.1f}s",
            f"Throughput:         {self.throughput_tps:.0f} tokens/s",
            f"Request rate:       {self.request_rate:.1f} req/s",
            f"TTFT P50:           {self.ttft_percentile(50):.0f}ms",
            f"TTFT P99:           {self.ttft_percentile(99):.0f}ms",
            f"TBT P50:            {self.tbt_percentile(50):.0f}ms",
            f"TBT P99:            {self.tbt_percentile(99):.0f}ms",
            f"E2E P50:            {self.e2e_percentile(50):.0f}ms",
            f"E2E P99:            {self.e2e_percentile(99):.0f}ms",
        ]
        return "\n".join(lines)


# Demonstrate the metrics on a simple example
example = RequestMetrics(
    req_id=0, prompt_len=50, output_len=100,
    submit_time_ms=0,
    first_token_time_ms=120,   # 120ms prefill + queue
    last_token_time_ms=3120,   # 3 seconds total
)
print(f"Example request:")
print(f"  TTFT:               {example.ttft_ms:.0f}ms")
print(f"  TBT:                {example.tbt_ms:.1f}ms/token  ({1000/example.tbt_ms:.0f} tokens/s)")
print(f"  E2E latency:        {example.e2e_latency_ms:.0f}ms  ({example.e2e_latency_ms/1000:.1f}s)")
print(f"  Normalized latency: {example.normalized_latency:.1f}ms/output_token")

## Part 2: Synthetic Serving Simulator

We'll build a simulator that models a realistic LLM serving system — hardware parameters,
request arrivals, prefill/decode costs — and produces timing traces we can analyze.

In [ ]:
@dataclass
class HardwareConfig:
    """Simplified hardware model for LLM inference timing."""
    model_params_b: float = 7.0          # model size in billions
    gpu_memory_bandwidth_tbs: float = 2.0 # TB/s (A10G: 0.6, A100: 2.0, H100: 3.35)
    gpu_flops_bf16_tflops: float = 312   # TFLOPS BF16 (A100 SXM)
    n_gpus: int = 1
    bytes_per_param: float = 2.0         # BF16
    
    @property
    def model_memory_gb(self) -> float:
        return self.model_params_b * 1e9 * self.bytes_per_param / 1e9
    
    def decode_latency_ms(self) -> float:
        """Time for one decode step (memory-bandwidth bound)."""
        effective_bw_gbs = self.gpu_memory_bandwidth_tbs * 1000 * self.n_gpus
        return self.model_memory_gb / effective_bw_gbs * 1000
    
    def prefill_latency_ms(self, n_tokens: int, batch_size: int = 1) -> float:
        """Time for prefill (compute-bound for long sequences)."""
        # FLOPs for attention + MLP in prefill ~ 2 * params * n_tokens
        flops = 2 * self.model_params_b * 1e9 * n_tokens
        compute_time_ms = flops / (self.gpu_flops_bf16_tflops * 1e12 * self.n_gpus) * 1000
        # Prefill is compute-bound; takes max of compute or bandwidth time
        bw_time_ms = self.decode_latency_ms()  # loading weights
        return max(compute_time_ms, bw_time_ms)


# Show timing breakdown for different model sizes
print("Decode latency (one token, one request, BF16) — memory bandwidth bound:")
print(f"{'Model':<15} {'Mem (GB)':>10} {'A10G':>10} {'A100':>10} {'H100':>10}")
print("-" * 55)
for params in [7, 13, 34, 70]:
    configs = [
        HardwareConfig(params, 0.6),   # A10G
        HardwareConfig(params, 2.0),   # A100
        HardwareConfig(params, 3.35),  # H100
    ]
    mem = configs[0].model_memory_gb
    lats = [f"{c.decode_latency_ms():.0f}ms" for c in configs]
    print(f"{params}B{'':<11} {mem:>10.0f} {lats[0]:>10} {lats[1]:>10} {lats[2]:>10}")

In [ ]:
def simulate_serving(
    hw: HardwareConfig,
    n_requests: int = 100,
    arrival_rate_rps: float = 2.0,    # requests per second
    max_batch_size: int = 8,
    prompt_len_mean: int = 100,
    output_len_mean: int = 150,
    output_len_std: int = 100,
    seed: int = 42,
) -> ServingMetrics:
    """
    Simulate LLM serving with continuous batching and realistic timing.
    Returns ServingMetrics with TTFT, TBT, and E2E measurements.
    """
    rng = np.random.RandomState(seed)
    
    # Generate requests
    inter_arrivals = rng.exponential(1000.0 / arrival_rate_rps, n_requests)  # ms
    arrive_times = np.cumsum(inter_arrivals)
    prompt_lens = rng.randint(prompt_len_mean // 2, prompt_len_mean * 2, n_requests)
    output_lens = np.clip(
        rng.normal(output_len_mean, output_len_std, n_requests).astype(int),
        10, output_len_mean * 4
    )
    
    decode_ms = hw.decode_latency_ms()  # per-token decode time
    
    # Simulate continuous batching
    @dataclass
    class Req:
        req_id: int
        arrive_ms: float
        prompt_len: int
        output_len: int
        start_ms: float = 0.0
        first_token_ms: float = 0.0
        last_token_ms: float = 0.0
        tokens_gen: int = 0
        
        @property
        def done(self): return self.tokens_gen >= self.output_len
    
    reqs = [Req(i, arrive_times[i], prompt_lens[i], output_lens[i]) for i in range(n_requests)]
    waiting = list(reqs)
    w_idx = 0
    active: list = []
    
    current_ms = 0.0
    
    while w_idx < len(waiting) or active:
        # Evict finished
        active = [r for r in active if not r.done]
        
        # Admit new requests
        while w_idx < len(waiting) and len(active) < max_batch_size and waiting[w_idx].arrive_ms <= current_ms:
            r = waiting[w_idx]
            r.start_ms = current_ms
            # Prefill: compute this request's prefill cost
            prefill_ms = hw.prefill_latency_ms(r.prompt_len)
            r.first_token_ms = current_ms + prefill_ms
            active.append(r)
            w_idx += 1
        
        if not active:
            if w_idx < len(waiting):
                current_ms = waiting[w_idx].arrive_ms
            continue
        
        # One decode step (batch processes in parallel)
        current_ms += decode_ms
        for r in active:
            r.tokens_gen += 1
            if r.done:
                r.last_token_ms = current_ms
    
    # Build metrics
    result = ServingMetrics(total_duration_ms=current_ms)
    for r in reqs:
        if r.last_token_ms > 0:
            result.requests.append(RequestMetrics(
                req_id=r.req_id,
                prompt_len=r.prompt_len,
                output_len=r.output_len,
                submit_time_ms=r.arrive_ms,
                first_token_time_ms=r.first_token_ms,
                last_token_time_ms=r.last_token_ms,
            ))
    
    return result


# Run a baseline simulation
hw_a100_7b = HardwareConfig(model_params_b=7, gpu_memory_bandwidth_tbs=2.0)

print(f"Hardware: 7B model, A100")
print(f"  Decode latency: {hw_a100_7b.decode_latency_ms():.1f}ms/token")
print(f"  Prefill latency (100 tokens): {hw_a100_7b.prefill_latency_ms(100):.1f}ms")
print()

metrics = simulate_serving(hw_a100_7b, n_requests=200, arrival_rate_rps=3.0, max_batch_size=8)
print(metrics.summary())

## Part 3: The Latency-Throughput Frontier

The most important visualization in LLM serving: sweep over arrival rates (or batch sizes)
and plot TTFT vs throughput. This Pareto curve tells you what tradeoffs you're making.

In [ ]:
def sweep_arrival_rates(
    hw: HardwareConfig,
    arrival_rates: list[float],
    max_batch_size: int = 16,
    n_requests: int = 300,
) -> list[dict]:
    results = []
    for rate in arrival_rates:
        m = simulate_serving(hw, n_requests=n_requests, arrival_rate_rps=rate,
                             max_batch_size=max_batch_size)
        if len(m.requests) < 10:
            continue
        results.append({
            'arrival_rate': rate,
            'throughput_tps': m.throughput_tps,
            'ttft_p50': m.ttft_percentile(50),
            'ttft_p99': m.ttft_percentile(99),
            'tbt_p50': m.tbt_percentile(50),
            'tbt_p99': m.tbt_percentile(99),
            'e2e_p50': m.e2e_percentile(50),
            'e2e_p99': m.e2e_percentile(99),
        })
    return results


arrival_rates = [0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 10.0]

# Compare 7B vs 70B model
hw_7b = HardwareConfig(7, 2.0)
hw_70b = HardwareConfig(70, 2.0, n_gpus=4)  # 70B on 4x A100

results_7b = sweep_arrival_rates(hw_7b, arrival_rates)
results_70b = sweep_arrival_rates(hw_70b, arrival_rates)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

def plot_frontier(ax, results, label, color):
    tps = [r['throughput_tps'] for r in results]
    ttft_p50 = [r['ttft_p50'] for r in results]
    ttft_p99 = [r['ttft_p99'] for r in results]
    ax.plot(tps, ttft_p50, 'o-', color=color, label=f"{label} P50", linewidth=2)
    ax.plot(tps, ttft_p99, 's--', color=color, label=f"{label} P99", alpha=0.7, linewidth=1.5)
    ax.fill_between(tps, ttft_p50, ttft_p99, alpha=0.1, color=color)

ax = axes[0]
plot_frontier(ax, results_7b, '7B (1×A100)', '#2ecc71')
plot_frontier(ax, results_70b, '70B (4×A100)', '#e74c3c')
ax.set_xlabel("Throughput (tokens/s)", fontsize=11)
ax.set_ylabel("TTFT (ms)", fontsize=11)
ax.set_title("Latency-Throughput Frontier\n(TTFT vs throughput)", fontsize=11)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# TBT vs throughput
ax2 = axes[1]
for results, label, color in [(results_7b, '7B', '#2ecc71'), (results_70b, '70B (4×A100)', '#e74c3c')]:
    tps = [r['throughput_tps'] for r in results]
    tbt_p50 = [r['tbt_p50'] for r in results]
    tbt_p99 = [r['tbt_p99'] for r in results]
    ax2.plot(tps, tbt_p50, 'o-', color=color, label=f"{label} P50", linewidth=2)
    ax2.plot(tps, tbt_p99, 's--', color=color, alpha=0.7, linewidth=1.5, label=f"{label} P99")

ax2.axhline(30, color='black', linestyle=':', label='30ms (streaming feels smooth)')
ax2.set_xlabel("Throughput (tokens/s)", fontsize=11)
ax2.set_ylabel("TBT (ms/token)", fontsize=11)
ax2.set_title("TBT vs Throughput", fontsize=11)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# Arrival rate vs throughput (shows saturation)
ax3 = axes[2]
for results, label, color in [(results_7b, '7B', '#2ecc71'), (results_70b, '70B (4×A100)', '#e74c3c')]:
    rates = [r['arrival_rate'] for r in results]
    tps = [r['throughput_tps'] for r in results]
    ax3.plot(rates, tps, 'o-', color=color, label=label, linewidth=2)

# Ideal (linear) reference
ideal_rates = np.array(arrival_rates)
ax3.plot(ideal_rates, ideal_rates * 150, '--', color='gray', alpha=0.5, label='Ideal (linear)')
ax3.set_xlabel("Arrival Rate (req/s)", fontsize=11)
ax3.set_ylabel("Throughput (tokens/s)", fontsize=11)
ax3.set_title("Throughput Saturation", fontsize=11)
ax3.legend(fontsize=8)
ax3.grid(alpha=0.3)

plt.suptitle("LLM Serving Metrics: 7B vs 70B on A100", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('latency_throughput_frontier.png', dpi=120, bbox_inches='tight')
plt.show()

## Part 4: Why Percentiles Matter More Than Means

Average latency is nearly useless for understanding user experience. The 1% of requests
that take 10x longer drive user churn, SLA violations, and timeout cascades.

This section shows *why* tail latency is so different from mean latency in LLM serving,
and how to interpret percentile distributions correctly.

In [ ]:
# Run a detailed simulation and analyze the full distribution
metrics_detail = simulate_serving(
    hw_7b, n_requests=500, arrival_rate_rps=3.0, max_batch_size=8
)

ttfts = [r.ttft_ms for r in metrics_detail.requests]
tbts = [r.tbt_ms for r in metrics_detail.requests if r.tbt_ms is not None]
e2es = [r.e2e_latency_ms for r in metrics_detail.requests]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

percentiles = [50, 75, 90, 95, 99, 99.9]
colors = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c', '#9b59b6', '#1abc9c']

for idx, (data, name, ax_row) in enumerate([
    (ttfts, 'TTFT (ms)', 0),
    (tbts, 'TBT (ms/token)', 0),
    (e2es, 'E2E Latency (ms)', 0),
]):
    ax = axes[0][idx]
    ax.hist(data, bins=50, color='#3498db', alpha=0.7, edgecolor='white', density=True)
    for p, c in zip(percentiles, colors):
        pval = np.percentile(data, p)
        ax.axvline(pval, color=c, linewidth=1.5, linestyle='--', label=f"P{p}: {pval:.0f}")
    ax.set_xlabel(name)
    ax.set_ylabel("Density")
    ax.set_title(f"{name} Distribution")
    ax.legend(fontsize=7)

# Percentile plots (the "needle" view)
ps = np.linspace(50, 99.9, 200)
for idx, (data, name) in enumerate([
    (ttfts, 'TTFT'),
    (tbts, 'TBT'),
    (e2es, 'E2E'),
]):
    ax = axes[1][idx]
    vals = [np.percentile(data, p) for p in ps]
    ax.plot(ps, vals, color='#e74c3c', linewidth=2)
    ax.axvline(99, color='gray', linestyle=':', alpha=0.7)
    ax.set_xlabel("Percentile")
    ax.set_ylabel(f"{name} (ms)")
    ax.set_title(f"{name} Percentile Curve")
    ax.grid(alpha=0.3)
    
    mean_val = np.mean(data)
    p99_val = np.percentile(data, 99)
    ax.text(0.05, 0.95, f"Mean: {mean_val:.0f}ms\nP99: {p99_val:.0f}ms\nP99/Mean: {p99_val/mean_val:.1f}x",
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle("Latency Distribution Analysis — 7B Model, 3 req/s, batch=8", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('latency_percentiles.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\nKey stat: P99/Mean ratio")
print(f"  TTFT P99/Mean:  {np.percentile(ttfts, 99)/np.mean(ttfts):.1f}x")
print(f"  TBT P99/Mean:   {np.percentile(tbts, 99)/np.mean(tbts):.1f}x")
print(f"  E2E P99/Mean:   {np.percentile(e2es, 99)/np.mean(e2es):.1f}x")
print("\n→ P99 can be 5-10x the mean. Optimizing for mean latency misses the user impact.")

## Part 5: Statistical Confidence Intervals for Metrics

A single benchmark run produces point estimates. But how stable are they? Running with
100 requests vs 1000 requests gives very different confidence. Bootstrap CIs let you
report "P99 TTFT = 450ms ± 80ms (95% CI)" instead of just "450ms".

In [ ]:
def bootstrap_ci(
    data: list[float],
    statistic_fn,
    n_bootstrap: int = 1000,
    confidence: float = 0.95,
) -> tuple[float, float, float]:
    """
    Bootstrap confidence interval for a statistic.
    Returns (estimate, lower_bound, upper_bound).
    """
    data = np.array(data)
    n = len(data)
    bootstrap_stats = [
        statistic_fn(np.random.choice(data, size=n, replace=True))
        for _ in range(n_bootstrap)
    ]
    alpha = 1 - confidence
    lo = np.percentile(bootstrap_stats, 100 * alpha / 2)
    hi = np.percentile(bootstrap_stats, 100 * (1 - alpha / 2))
    return statistic_fn(data), lo, hi


# Show how CI width varies with sample size
print("Bootstrap 95% CI for P99 TTFT vs sample size:")
print(f"{'N Requests':>12} {'P99 Estimate':>14} {'95% CI':>22} {'CI Width':>10}")
print("-" * 62)

for n_req in [20, 50, 100, 200, 500]:
    m = simulate_serving(hw_7b, n_requests=n_req, arrival_rate_rps=3.0, seed=42)
    ttfts_n = [r.ttft_ms for r in m.requests]
    est, lo, hi = bootstrap_ci(ttfts_n, lambda x: np.percentile(x, 99))
    print(f"{n_req:>12} {est:>14.0f} [{lo:>8.0f}, {hi:>8.0f}] {hi-lo:>10.0f}ms")

# Visualize CI convergence
sample_sizes = [10, 20, 50, 100, 200, 500, 1000]
ci_widths = []
estimates = []

for n in sample_sizes:
    m = simulate_serving(hw_7b, n_requests=n, arrival_rate_rps=3.0)
    ttfts_n = [r.ttft_ms for r in m.requests]
    est, lo, hi = bootstrap_ci(ttfts_n, lambda x: np.percentile(x, 99), n_bootstrap=500)
    ci_widths.append(hi - lo)
    estimates.append(est)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(sample_sizes, ci_widths, 'o-', color='#e74c3c', linewidth=2)
ax.set_xlabel("Number of Requests in Benchmark")
ax.set_ylabel("95% CI Width (ms)")
ax.set_title("CI Width vs Sample Size (P99 TTFT)")
ax.set_xscale('log')
ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(sample_sizes, estimates, 'o-', color='#2ecc71', linewidth=2, label='P99 estimate')
true_p99 = np.percentile([r.ttft_ms for r in simulate_serving(hw_7b, 5000, 3.0).requests], 99)
ax2.axhline(true_p99, color='black', linestyle='--', label=f'True P99 ≈ {true_p99:.0f}ms')
ax2.set_xlabel("Number of Requests")
ax2.set_ylabel("P99 TTFT (ms)")
ax2.set_title("Estimate Stability vs Sample Size")
ax2.set_xscale('log')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('ci_convergence.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nRule of thumb: need ~200+ requests for stable P99 estimates. P99.9 requires ~2000+.")

## Part 6: `benchpress` Prototype — Real Measurement Harness

Bringing it together: a structured measurement harness that could be adapted to test
any real LLM endpoint. This is the foundation of the `benchpress` tool.

In [ ]:
class BenchpressHarness:
    """
    Structured benchmark harness for LLM serving.
    
    In production, replace `_simulate_request` with actual API calls.
    The harness handles timing, concurrency management, and metric aggregation.
    """
    def __init__(self, hw_config: HardwareConfig, max_batch_size: int = 8):
        self.hw = hw_config
        self.max_batch_size = max_batch_size
        self.results: list[RequestMetrics] = []
    
    def run_benchmark(
        self,
        n_warmup: int = 10,
        n_requests: int = 200,
        arrival_rate_rps: float = 3.0,
        prompt_len_mean: int = 100,
        output_len_mean: int = 150,
        n_bootstrap: int = 1000,
    ) -> dict:
        """Run a complete benchmark and return results with CIs."""
        print(f"Running warmup ({n_warmup} requests)...", end=' ', flush=True)
        _ = simulate_serving(self.hw, n_warmup, arrival_rate_rps, self.max_batch_size,
                             prompt_len_mean, output_len_mean)
        print("done")
        
        print(f"Running benchmark ({n_requests} requests at {arrival_rate_rps} req/s)...", end=' ', flush=True)
        metrics = simulate_serving(self.hw, n_requests, arrival_rate_rps, self.max_batch_size,
                                   prompt_len_mean, output_len_mean)
        print("done")
        
        ttfts = [r.ttft_ms for r in metrics.requests]
        tbts = [r.tbt_ms for r in metrics.requests if r.tbt_ms is not None]
        e2es = [r.e2e_latency_ms for r in metrics.requests]
        
        results = {
            'n_requests': len(metrics.requests),
            'duration_s': metrics.total_duration_ms / 1000,
            'throughput_tps': metrics.throughput_tps,
            'request_rate_rps': metrics.request_rate,
        }
        
        for data, name in [(ttfts, 'ttft'), (tbts, 'tbt'), (e2es, 'e2e')]:
            for p in [50, 90, 99]:
                est, lo, hi = bootstrap_ci(data, lambda x, p=p: np.percentile(x, p), n_bootstrap)
                results[f'{name}_p{p}_ms'] = est
                results[f'{name}_p{p}_ci_lo'] = lo
                results[f'{name}_p{p}_ci_hi'] = hi
        
        return results
    
    def print_report(self, results: dict):
        print("\n" + "═" * 60)
        print(f"  BENCHPRESS REPORT")
        print("═" * 60)
        print(f"  Requests:           {results['n_requests']}")
        print(f"  Duration:           {results['duration_s']:.1f}s")
        print(f"  Throughput:         {results['throughput_tps']:.0f} tokens/s")
        print(f"  Request rate:       {results['request_rate_rps']:.1f} req/s")
        print()
        print(f"  {'Metric':<18} {'P50':>10} {'P90':>10} {'P99':>10}")
        print(f"  {'-'*50}")
        for metric, label in [('ttft', 'TTFT (ms)'), ('tbt', 'TBT (ms/tok)'), ('e2e', 'E2E (ms)')]:
            p50 = results[f'{metric}_p50_ms']
            p90 = results[f'{metric}_p90_ms']
            p99 = results[f'{metric}_p99_ms']
            p99_lo = results[f'{metric}_p99_ci_lo']
            p99_hi = results[f'{metric}_p99_ci_hi']
            print(f"  {label:<18} {p50:>10.0f} {p90:>10.0f} {p99:>10.0f}")
            print(f"  {'  P99 95% CI':<18} {'':>10} {'':>10} [{p99_lo:>6.0f}, {p99_hi:>6.0f}]")
        print("═" * 60)


harness = BenchpressHarness(hw_a100_7b, max_batch_size=8)
results = harness.run_benchmark(n_warmup=10, n_requests=300, arrival_rate_rps=3.0)
harness.print_report(results)

## Summary

LLM serving metrics are more subtle than traditional web service metrics because the
"response" is a variable-length token stream, not a single value.

### Key Takeaways

**The Three Metrics**  
- **TTFT** (Time to First Token): User-facing interactivity. Dominated by prefill + queuing.
- **TBT** (Time Between Tokens): Streaming smoothness. Dominated by decode phase latency.
- **Throughput** (tokens/s): Operator metric. Maximized by batching.

**The Core Tradeoff**  
Throughput and TTFT are fundamentally in tension. More concurrent requests = higher GPU
utilization = higher throughput, but also longer queuing = worse TTFT. Every production
system chooses a point on this Pareto frontier based on its SLA.

**Always Report Percentiles, Not Means**  
P99 TTFT is often 5–10× the mean. Optimizing for mean latency misses the tail that drives
user complaints. P99 is the SLA metric; P99.9 is the on-call metric.

**Statistical Rigor**  
Point estimates from small samples are noisy. Bootstrap confidence intervals are cheap
to compute and essential for any benchmark you intend to publish or compare against.
Rule of thumb: 200+ requests for stable P99, 2000+ for P99.9.

### What's Next (Notebook 13: Running Local Models)

Now that we understand the metrics, let's actually measure them on real hardware. Notebook 13
covers llama.cpp, Ollama, and MLX — the three main options for running models locally on
Apple Silicon — and uses this benchpress harness to compare their real-world performance.